# Topic 5: Production Deployment

> **Phase 5 → Week 9 → Topic 5**

---

## From Notebook to Production

```
Notebook (development)          Production Pipeline
──────────────────────          ────────────────────────────────────
spark.read.csv(...)             spark-submit my_etl.py --date 2024-01-15
Manual cell-by-cell runs   →    Scheduled by Airflow/Glue/Databricks
Hardcoded paths                 Config from environment variables / args
No error handling               try/except + dead letter + alerts
InferSchema=True                Explicit StructType schema
show() for exploration          Structured JSON logging
Local SparkSession              YARN / Kubernetes / EMR / Databricks
```

---

## Interview Cheat Sheet

**Q: How do you submit a Spark job to a cluster?**
> `spark-submit` is the standard CLI. Key flags: `--master yarn` (or `k8s://`, `spark://`), `--deploy-mode cluster` (driver on cluster) vs `client` (driver on submitting machine), `--executor-memory`, `--executor-cores`, `--num-executors`, `--conf spark.key=val`, `--py-files dependencies.zip`, `--jars extra.jar`. In cloud: Databricks uses job clusters with `dbutils.widgets`; AWS uses Glue or EMR Step API.

**Q: What is deploy-mode client vs cluster?**
> `client`: driver runs on the machine that calls `spark-submit`. Logs stream to your terminal. Good for interactive/debugging. `cluster`: driver runs on a random worker node in the cluster. More resilient (if submitting machine dies, job continues). Required for production batch jobs on YARN/K8s where you submit and disconnect.

**Q: How do you pass configuration to a production Spark job?**
> Three approaches: (1) `argparse` — parse CLI args (`--date 2024-01-15`, `--env prod`). (2) Environment variables — read with `os.environ.get()`. (3) Config files — YAML/JSON loaded at startup. In practice: combine argparse for job-specific params (date, input path) + environment variables for secrets (DB password) + a YAML config file for cluster-level settings.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import argparse, os, sys, json
from datetime import date, timedelta

spark = SparkSession.builder \
    .appName("Week9 - Production Deployment") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Production-Ready Script Structure

In [ ]:
# Write a production-ready ETL script to /tmp
script = '''
#!/usr/bin/env python3
"""
orders_daily_etl.py  —  Daily orders ETL pipeline

Usage:
    spark-submit orders_daily_etl.py --date 2024-01-15 --env prod
"""
import argparse
import logging
import os
import sys
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, IntegerType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s"
)
logger = logging.getLogger("orders_etl")


# ── Configuration ────────────────────────────────────────────────────────────

CONFIGS = {
    "dev":  {"input":  "/workspace/week4/data",
             "output": "/tmp/etl_output/dev",
             "partitions": 4},
    "prod": {"input":  "s3://my-data-lake/raw/orders",
             "output": "s3://my-data-lake/processed/orders",
             "partitions": 200},
}

ORDERS_SCHEMA = StructType([
    StructField("order_id",    StringType(),  True),
    StructField("customer_id", StringType(),  True),
    StructField("product_id",  StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("order_date",  DateType(),    True),
    StructField("status",      StringType(),  True),
    StructField("amount",      DoubleType(),  True),
])


# ── Transformations ──────────────────────────────────────────────────────────

def filter_valid_orders(df):
    return df.filter(
        F.col("amount").isNotNull() &
        (F.col("amount") > 0) &
        F.col("customer_id").isNotNull()
    )

def add_order_tier(df):
    return df.withColumn("tier",
        F.when(F.col("amount") > 500, "High")
         .when(F.col("amount") > 100, "Medium")
         .otherwise("Low")
    )


# ── Main pipeline ────────────────────────────────────────────────────────────

def run(args):
    cfg = CONFIGS[args.env]

    spark = SparkSession.builder \\
        .appName(f"orders_daily_etl_{args.date}") \\
        .config("spark.sql.shuffle.partitions", str(cfg["partitions"])) \\
        .getOrCreate()

    logger.info(f"Starting ETL for date={args.date} env={args.env}")

    # Read
    raw = spark.read \\
        .option("header", "true") \\
        .option("mode", "PERMISSIVE") \\
        .schema(ORDERS_SCHEMA) \\
        .csv(f"{cfg[\'input\']}/orders.csv")

    input_count = raw.count()
    logger.info(f"Read {input_count} raw rows")

    # Transform
    valid   = filter_valid_orders(raw)
    tiered  = add_order_tier(valid)

    output_count = tiered.count()
    rejected     = input_count - output_count
    logger.info(f"Output {output_count} rows ({rejected} rejected, {rejected/input_count*100:.1f}%)")

    if rejected / input_count > 0.1:
        logger.warning(f"HIGH rejection rate: {rejected/input_count*100:.1f}%")

    # Write
    out_path = f"{cfg[\'output\']}/dt={args.date}"
    tiered.write.mode("overwrite").parquet(out_path)
    logger.info(f"Written to {out_path}")

    spark.stop()
    return 0


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Daily orders ETL")
    parser.add_argument("--date", required=True, help="Processing date YYYY-MM-DD")
    parser.add_argument("--env",  default="dev", choices=["dev", "prod"])
    args = parser.parse_args()
    sys.exit(run(args))
'''

with open("/tmp/orders_daily_etl.py", "w") as f:
    f.write(script)

print("Production script written to /tmp/orders_daily_etl.py")
print()
print("Key structure:")
print("  1. CONFIGS dict    — env-specific settings (dev/prod)")
print("  2. Explicit schema — no inferSchema")
print("  3. Pure transform functions — testable separately")
print("  4. Structured logging — INFO/WARN/ERROR with context")
print("  5. argparse main()  — all params from CLI, not hardcoded")
print("  6. sys.exit(code)   — exit code 0=success, 1=failure for schedulers")

## Part 2: spark-submit Reference

In [ ]:
print("""
spark-submit Command Reference:
════════════════════════════════════════════════════════════════
# Minimal (local mode)
spark-submit my_script.py --date 2024-01-15

# Full production command (YARN cluster)
spark-submit \\
  --master yarn \\
  --deploy-mode cluster \\
  --name "orders_daily_etl" \\
  --num-executors 10 \\
  --executor-cores 4 \\
  --executor-memory 8g \\
  --driver-memory 4g \\
  --conf spark.sql.shuffle.partitions=200 \\
  --conf spark.sql.adaptive.enabled=true \\
  --conf "spark.executor.extraJavaOptions=-XX:+UseG1GC" \\
  --py-files dependencies.zip \\
  --files config.yaml \\
  orders_daily_etl.py \\
  --date 2024-01-15 --env prod

Key flags:
  --master          yarn | k8s://apiserver | spark://host:7077 | local[N]
  --deploy-mode     cluster (driver on cluster) | client (driver local)
  --num-executors   N static executors (or use dynamic allocation)
  --executor-cores  vCPUs per executor (4-5 recommended)
  --executor-memory JVM heap per executor (8-16g recommended)
  --driver-memory   Driver JVM heap (2-4g for most jobs)
  --conf            Any Spark config key=value
  --py-files        Extra Python files/zips to distribute to executors
  --files           Arbitrary files to distribute (config, models)
  --jars            Extra JARs (Delta Lake, JDBC drivers, etc.)
  --packages        Maven coordinates (downloads automatically)

Deploy-mode difference:
  client:  driver on submitting machine → logs stream to terminal
           Good for: interactive dev, debugging
  cluster: driver on cluster node → disconnect-safe
           Required for: production, long-running jobs, unattended runs
════════════════════════════════════════════════════════════════
""")

## Part 3: Airflow Integration Pattern

In [ ]:
print("""
Apache Airflow DAG for Daily Spark ETL:
════════════════════════════════════════════════════════════════
from airflow import DAG
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

DEFAULT_ARGS = {
    'owner': 'data-engineering',
    'start_date': datetime(2024, 1, 1),
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'email_on_failure': True,
    'email': ['alerts@company.com'],
}

with DAG(
    dag_id='orders_daily_etl',
    default_args=DEFAULT_ARGS,
    schedule_interval='0 2 * * *',  # 2 AM daily
    catchup=False,
    tags=['etl', 'orders'],
) as dag:

    run_etl = SparkSubmitOperator(
        task_id='run_orders_etl',
        application='s3://my-bucket/jobs/orders_daily_etl.py',
        conn_id='spark_cluster',
        name='orders_daily_{{ ds }}',           # ds = execution date YYYY-MM-DD
        executor_memory='8g',
        executor_cores=4,
        num_executors=10,
        application_args=['--date', '{{ ds }}', '--env', 'prod'],
        conf={
            'spark.sql.shuffle.partitions': '200',
            'spark.sql.adaptive.enabled': 'true',
        },
    )

Key Airflow patterns:
  {{ ds }}           → execution date (YYYY-MM-DD), passed to your script
  retries=3          → auto-retry on failure (your script must be idempotent!)
  catchup=False      → don't backfill missed runs on startup
  email_on_failure   → alerts on failure
  depends_on_past    → only run if yesterday's run succeeded
════════════════════════════════════════════════════════════════
""")

## Part 4: Cloud Deployment Patterns

In [ ]:
print("""
Cloud Spark Deployment Options:
════════════════════════════════════════════════════════════════
Platform       Managed  Cost      Best For
─────────────  ───────  ───────   ────────────────────────────────
AWS EMR        ✓        Medium    YARN-managed Spark on EC2
AWS Glue       ✓✓       Higher    Serverless Spark, no cluster mgmt
Databricks     ✓✓✓      High      Best DX, Unity Catalog, Delta native
GCP Dataproc   ✓        Medium    GCP-native YARN Spark
Azure Synapse  ✓✓       High      Azure-integrated Spark + SQL
Self-managed   ✗        Lower     Full control, more ops overhead

AWS EMR — Typical Setup:
  # Step (job) submission
  aws emr add-steps --cluster-id j-XXXXX \\
    --steps Type=Spark,Name='orders-etl',\\
            Args=[--deploy-mode,cluster,\\
                  s3://bucket/jobs/orders_etl.py,\\
                  --date,2024-01-15,--env,prod]

  # Or use EMR Serverless (no cluster to manage)
  aws emr-serverless start-job-run \\
    --application-id app-id \\
    --execution-role-arn arn:aws:iam::... \\
    --job-driver '{"sparkSubmit": {"entryPoint": "s3://bucket/jobs/etl.py"}}'

Databricks — Job submission:
  # Via Databricks CLI
  databricks jobs run-now --job-id 12345 \\
    --python-named-params '{"date": "2024-01-15"}'

  # Or REST API / Terraform for job definitions
════════════════════════════════════════════════════════════════
""")

## Part 5: Packaging Dependencies

In [ ]:
print("""
Packaging Python Dependencies for spark-submit:
════════════════════════════════════════════════════════════════
Problem: your script imports custom modules (transforms.py, utils.py).
         Executors don't have access to your local filesystem.

Solution 1: --py-files (simple, for small codebases)
  # Bundle all .py files into a zip
  zip dependencies.zip transforms.py utils.py config.py

  spark-submit --py-files dependencies.zip main.py

  # In your code:
  # Spark automatically adds dependencies.zip to sys.path
  from transforms import filter_valid_orders  # works on all executors!

Solution 2: Python package + virtualenv (recommended for larger projects)
  # 1. Package your code
  python setup.py bdist_egg  # creates dist/mypackage-1.0.0-py3.11.egg

  # 2. Submit with egg
  spark-submit --py-files dist/mypackage-1.0.0-py3.11.egg main.py

Solution 3: Docker container (Kubernetes / Databricks)
  # Build container with all dependencies pre-installed
  FROM apache/spark:3.5.3
  COPY mypackage/ /opt/mypackage/
  RUN pip install -r requirements.txt

  # Submit using the container image
  spark-submit --master k8s://... \\
    --conf spark.kubernetes.container.image=my-registry/spark-etl:v1.2 \\
    main.py

Solution 4: Pre-install on cluster (YARN cluster)
  # Bootstrap action / cluster init script installs packages once
  pip install mypackage==1.0.0
  # All jobs on this cluster can import mypackage
════════════════════════════════════════════════════════════════
""")

## Part 6: Production Checklist

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         PRODUCTION ETL DEPLOYMENT CHECKLIST                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CODE QUALITY                                                ║
║  □ No hardcoded paths/passwords (use argparse + env vars)    ║
║  □ Explicit StructType schemas (no inferSchema)              ║
║  □ Error handling in all UDFs (try/except → null)            ║
║  □ Dead letter table for bad records                         ║
║  □ Structured JSON logging at key checkpoints                ║
║  □ sys.exit(0/1) for scheduler integration                   ║
║                                                              ║
║  IDEMPOTENCY                                                 ║
║  □ Write uses overwrite/replaceWhere/MERGE (not append)      ║
║  □ Re-running the job with same date = same result           ║
║  □ Watermark updated atomically with write                   ║
║                                                              ║
║  DATA QUALITY                                                ║
║  □ Row count validation (input, filtered, output)            ║
║  □ Rejection rate alert (> 5% = warning)                     ║
║  □ Row count drop alert (> 20% vs yesterday = alert)         ║
║  □ Zero-row output alert                                     ║
║                                                              ║
║  PERFORMANCE                                                 ║
║  □ AQE enabled                                               ║
║  □ shuffle.partitions = 2-4× executor cores                  ║
║  □ Broadcast small dimension tables                          ║
║  □ Filter and project early                                   ║
║  □ G1GC enabled for executors                                ║
║                                                              ║
║  TESTING                                                     ║
║  □ Unit tests for all transformation functions               ║
║  □ Edge cases: empty input, all nulls, boundary values       ║
║  □ Tests run in CI/CD on every PR                            ║
║                                                              ║
║  OPERATIONS                                                  ║
║  □ Job is scheduled (Airflow/Glue/Databricks Jobs)           ║
║  □ Failure alerts configured (email/Slack/PagerDuty)         ║
║  □ Output location documented and versioned (Delta)          ║
║  □ VACUUM scheduled for Delta tables                         ║
║  □ OPTIMIZE scheduled for Delta tables (post large writes)   ║
╚══════════════════════════════════════════════════════════════╝
""")

## Exercises

1. Convert the orders ETL notebook from Week 8 into a production-ready `orders_etl.py` script with argparse, structured logging, explicit schema, and dead letter handling.
2. Write a `spark-submit` command for the above script targeting YARN cluster with: 5 executors, 4 cores each, 12GB memory, AQE enabled, G1GC enabled.
3. What is the difference between `--deploy-mode client` and `--deploy-mode cluster`? Which would you use for a midnight batch job and why?
4. Your job fails at 3 AM and Airflow retries it automatically at 3:05 AM. The original job had already written 50% of the output before failing. With idempotent design (`replaceWhere`), what happens on the retry?
5. List 5 differences between writing ETL for a Jupyter notebook vs writing it for production deployment.

In [ ]:
# Exercise 2: spark-submit command
print("""
spark-submit command for Exercise 2:

spark-submit \\
  --master yarn \\
  --deploy-mode cluster \\
  --name "orders_etl_$(date +%Y-%m-%d)" \\
  --num-executors 5 \\
  --executor-cores 4 \\
  --executor-memory 12g \\
  --driver-memory 4g \\
  --conf spark.sql.adaptive.enabled=true \\
  --conf spark.sql.shuffle.partitions=80 \\
  --conf "spark.executor.extraJavaOptions=-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35" \\
  --conf spark.sql.adaptive.coalescePartitions.enabled=true \\
  --conf spark.sql.adaptive.skewJoin.enabled=true \\
  orders_etl.py \\
  --date $(date +%Y-%m-%d) --env prod

Note: shuffle.partitions = 5 executors × 4 cores × 4 = 80
"""
)

# Answer Q4: idempotent retry
print("""
Answer Q4: Idempotent retry with replaceWhere

Scenario: job wrote 50% output, failed, retried at 3:05 AM.

With replaceWhere (idempotent):
  The retry overwrites the ENTIRE partition for that date.
  The partial 50% output is replaced by the full correct output.
  Result: clean, correct data. No duplicates, no data loss.

Without idempotent design (plain append):
  Partial output (50%) remains from the failed run.
  Retry appends another full output (100%).
  Result: 150% of data — first 50% duplicated!
  This is why idempotency is non-negotiable for production ETL.
""")